# Comparación de la heurística KNN con la sintonización individual y con un controlador de parámetros fijos

En esta libreta se evalúa la heurística de estimación de hiperparámetros frente a dos referencias, simulando las tres estrategias con la misma planta virtual, la misma trayectoria de referencia Λ(t) y la misma implementación. El controlador opera en lazo cerrado durante la sedación y el mantenimiento; durante la recuperación, de 30 minutos, se desactiva y la planta evoluciona en lazo abierto, por lo que ξ y el RMSE se evalúan en las fases de sedación y mantenimiento:

- **AG:** parámetros sintonizados individualmente mediante algoritmo genético (`SINTONIZACIONES.csv`).
- **KNN:** parámetros estimados por la heurística en su variante global (K = 2), con el orden p de la distancia de Minkowski que minimizó ξ en `KNN_Estimacion_PID.ipynb` (archivo `Barrido_p_KNN.csv`).
- **Parámetros medianos:** un único controlador cuyos parámetros son la mediana, componente a componente, de los parámetros del espacio de referencia.

La comparación se efectúa sobre el conjunto de estimación y sobre el conjunto de casos con sintonización deficiente, es decir, los pacientes cuyo RMSE del AG constituye un valor atípico superior según el criterio de Tukey ($\mathrm{RMSE} > Q_3 + 1.5\,\mathrm{RIC}$).



## Etapa 1: Configuración del entorno (tomada de `KNN_Estimacion_PID.ipynb`)

In [1]:
# =============================================================================
# CONFIGURACIÓN GLOBAL 
# =============================================================================

# ── Rutas ─────────────────────────────────────────────────────────────────────
RUTA_MODELO          = r"C:\Users\yordi\Desktop\modeloBIS.pt"
RUTA_GRAFICAS        = r"C:\Users\yordi\Desktop"
RUTA_SINTONIZACIONES = "SINTONIZACIONES.csv"   # Archivo generado en la libreta previa

# ── Rangos de normalización del modelo predictor del BIS ──────────────────────
# (fijos, correspondientes al preprocesamiento con el que fue entrenado)
EDAD_MIN, EDAD_MAX     = 16.00, 90.00
PESO_MIN, PESO_MAX     = 34.10, 112.80
ALTURA_MIN, ALTURA_MAX = 140.20, 186.60

# =============================================================================
# IMPORTACIONES
# =============================================================================
import os, time, math, sys
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import torch
import torch.nn as nn
from scipy import stats
from Modelo_Predictor_BIS import ModeloAnestesia
from Normalizacion_BIS import (cargar_estadisticas, describir_estadisticas,
                               construir_ventanas_simulacion)

# Estadísticas globales de normalización, calculadas sobre el conjunto de
# entrenamiento por Crea_tensores_pytorch.ipynb; deben ser las mismas con
# las que se entrenó el modelo predictor
ESTADISTICAS = cargar_estadisticas()
print("[INFO] Estadísticas de normalización:")
print(describir_estadisticas(ESTADISTICAS))

# =============================================================================
# DISPOSITIVO Y CARGA DEL MODELO PREDICTOR DEL BIS
# =============================================================================
dispositivo = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[INFO] Dispositivo : {dispositivo}")
if dispositivo.type == "cuda":
    print(f"[INFO] GPU         : {torch.cuda.get_device_name(0)}")

modelo = ModeloAnestesia().to(dispositivo)
modelo.load_state_dict(
    torch.load(RUTA_MODELO, map_location=dispositivo, weights_only=True)
)
modelo.eval()
for p in modelo.parameters():
    p.requires_grad_(False)

print(f"[INFO] Modelo cargado — "
      f"{sum(p.numel() for p in modelo.parameters()):,} parámetros")


# =============================================================================
# NORMALIZACIÓN DE LAS COVARIABLES PARA EL MODELO PREDICTOR DEL BIS
# =============================================================================
# NOTA: esta normalización corresponde al preprocesamiento fijo del modelo
# predictor del BIS y es independiente de la normalización min-max que se
# aplicará posteriormente para el algoritmo de K vecinos más cercanos.

def normalizar_paciente(edad, peso, altura, sexo=0):
    """
    Convierte las características fisiológicas del paciente a los valores
    normalizados que espera el modelo predictor del BIS.

    Parámetros
    ----------
    edad, peso, altura : valores en unidades naturales (años, kg, cm)
    sexo               : 0 = masculino, 1 = femenino

    Retorna
    -------
    Tupla (edad_n, peso_n, altura_n, sexo_n)
    """
    return (
        (edad   - EDAD_MIN)   / (EDAD_MAX   - EDAD_MIN),
        (peso   - PESO_MIN)   / (PESO_MAX   - PESO_MIN),
        (altura - ALTURA_MIN) / (ALTURA_MAX - ALTURA_MIN),
        float(sexo),
    )


def hacer_cov_lote(df_pac: pd.DataFrame) -> torch.Tensor:
    """
    Construye un tensor de covariables (N, 4) a partir de un DataFrame de
    pacientes, lo que permite simular a toda una cohorte en un único lote,
    asignando a cada paciente sus propias características fisiológicas.

    Parámetros
    ----------
    df_pac : DataFrame con las columnas Edad, Peso, Altura y Sexo_num

    Retorna
    -------
    torch.Tensor de forma (N, 4) en el dispositivo activo
    """
    filas = [
        normalizar_paciente(float(f["Edad"]), float(f["Peso"]),
                             float(f["Altura"]), int(f["Sexo_num"]))
        for _, f in df_pac.iterrows()
    ]
    return torch.tensor(filas, dtype=torch.float32, device=dispositivo)


[INFO] Estadísticas de normalización:
  propofol      : media = 0.00512937, desviación estándar = 0.00783327
  remifentanilo : media = 0.00602943, desviación estándar = 0.00732605
  bis           : media = 44.9932, desviación estándar = 12.6622
[INFO] Dispositivo : cuda
[INFO] GPU         : NVIDIA GeForce RTX 5060
[INFO] Modelo cargado — 144,212 parámetros


## Etapa 2: Filtro de señales, trayectoria Λ(t) y función de error ξ (tomada de `KNN_Estimacion_PID.ipynb`)

In [2]:
# =============================================================================
# FILTRO DE MEDIA EXPONENCIAL MÓVIL, TRAYECTORIA Y FUNCIÓN DE ERROR
# =============================================================================

# ── Hiperparámetros del filtro de media exponencial móvil ────────────────────
# Antes de introducirse al modelo predictor del BIS, las señales de control se
# filtran mediante una media exponencial móvil aplicada sobre una ventana
# deslizante de longitud fija, rellenada con ceros en su parte inicial:
#
#       y[n] = α · u[n] + (1 − α) · y[n − 1]
#
# Los valores coinciden con los empleados en la sintonización poblacional
# (PID_Filtro_EMA.ipynb), con la que se generó SINTONIZACIONES.csv.
VENTANA_EMA = 12      # ← Longitud de la ventana deslizante
ALFA_EMA    = 0.05    # ← Factor de suavizado α


def construir_pesos_ema(ventana: int = VENTANA_EMA,
                         alfa: float = ALFA_EMA) -> torch.Tensor:
    """
    Construye el vector de pesos equivalente a la recursión de la media
    exponencial móvil sobre una ventana de longitud fija.

    Al reinicializarse la recursión con y[−1] = 0 al comienzo de cada ventana,
    la media exponencial móvil resulta ser una combinación lineal fija de las
    muestras contenidas en dicha ventana:

        y = Σ_{k=0}^{V−1}  α · (1 − α)^(V − 1 − k) · x[k]

    Expresarla de esta forma permite evaluarla mediante un único producto
    matricial, en lugar de un bucle recursivo, lo que reduce sustancialmente
    el tiempo de cómputo sin alterar en absoluto el resultado numérico.

    Parámetros
    ----------
    ventana : longitud de la ventana deslizante
    alfa    : factor de suavizado

    Retorna
    -------
    torch.Tensor de forma (ventana,) con los pesos del filtro
    """
    k = torch.arange(ventana, dtype=torch.float32, device=dispositivo)
    return alfa * (1.0 - alfa) ** (ventana - 1 - k)


@torch.no_grad()
def filtro_ema_ventana(buffer_crudo: torch.Tensor, t: int,
                        pesos_ema: torch.Tensor) -> torch.Tensor:
    """
    Calcula el valor filtrado en el instante t para todo el lote en paralelo.

    La ventana se conforma con las últimas V muestras de la señal cruda,
    completando con ceros las posiciones para las cuales aún no existe
    historial.

    Parámetros
    ----------
    buffer_crudo : (N, T_sim) — historial de la señal cruda de control
    t             : instante actual
    pesos_ema     : (V,) — pesos del filtro

    Retorna
    -------
    (N,) — valor filtrado en el instante t
    """
    N, V = buffer_crudo.shape[0], pesos_ema.shape[0]
    ini  = max(0, t - V + 1)
    fin  = t + 1
    n_relleno = V - (fin - ini)

    if n_relleno > 0:
        ceros = torch.zeros(N, n_relleno, device=buffer_crudo.device)
        ventana_datos = torch.cat([ceros, buffer_crudo[:, ini:fin]], dim=1)
    else:
        ventana_datos = buffer_crudo[:, ini:fin]

    return ventana_datos @ pesos_ema


# =============================================================================
# VENTANAS DE ENTRADA DEL MODELO PREDICTOR
# =============================================================================
# Las ventanas se construyen mediante construir_ventanas_simulacion, definida
# en Normalizacion_BIS.py, que aplica el mismo relleno y la misma
# normalización global empleados en el entrenamiento del modelo.


# =============================================================================
# TRAYECTORIA DE REFERENCIA Λ(t)  (SEDACIÓN · MANTENIMIENTO · RECUPERACIÓN)
# =============================================================================
#   Fase 1 — Sedación      (   0 –  300 s): Λ(t) = 98 + 56.8·tanh(−0.004·t)
#   Fase 2 — Mantenimiento ( 300 – 3900 s): Λ(t) = 50  (constante)
#   Fase 3 — Recuperación  (3900 – 5700 s): Λ(t) = 50 + 42·tanh(0.003·(t−3900))
#
# El PID opera en lazo cerrado durante las fases de sedación y mantenimiento.
# Al iniciar la fase de recuperación, el controlador se desactiva y la señal de
# control se anula, de modo que la planta evoluciona en lazo abierto durante
# 30 minutos; en esta fase Λ(t) se conserva únicamente como curva de
# comparación. La configuración coincide con la de la sintonización
# poblacional (PID_Filtro_EMA.ipynb).

T_SEDACION      = 300     # [s]
T_MANTENIMIENTO = 3600    # [s]
T_RECUPERACION  = 1800    # [s] — recuperación en lazo abierto (30 min)

SED_A, SED_B, SED_K = 98.0, 56.8, -0.004
BIS_MANT            = 50.0
REC_A, REC_B, REC_K = 50.0, 42.0, 0.003

DT       = 10.0    # Período de muestreo [s]
T_VEN    = 180     # Longitud de ventana del modelo [muestras]
PPF_MAX  = 0.05    # Saturación máxima de propofol (U_MAX de la sintonización)
REMI_MAX = 0.05    # Saturación máxima de remifentanilo (U_MAX de la sintonización)

N_SED = int(T_SEDACION      / DT)
N_MAN = int(T_MANTENIMIENTO / DT)
N_REC = int(T_RECUPERACION  / DT)
N_TOT = N_SED + N_MAN + N_REC
N_CTRL = N_SED + N_MAN    # Muestras con el controlador activo

LAMBDA_REF = np.zeros(N_TOT, dtype=np.float32)      # Λ(t)

for _i in range(N_SED):                                     # Fase 1
    LAMBDA_REF[_i] = SED_A + SED_B * np.tanh(SED_K * _i * DT)

LAMBDA_REF[N_SED : N_SED + N_MAN] = BIS_MANT                # Fase 2

for _i in range(N_REC):                                     # Fase 3
    LAMBDA_REF[N_SED + N_MAN + _i] = REC_A + REC_B * np.tanh(REC_K * _i * DT)

REF_TENSOR = torch.tensor(LAMBDA_REF, dtype=torch.float32, device=dispositivo)
EJE_T      = np.arange(N_TOT) * DT      # Eje temporal [s]
T_FINAL    = N_TOT * DT                 # Duración total de la trayectoria [s]
T_CONTROL  = N_CTRL * DT                # T_c, duración de sedación y mantenimiento [s]
PESOS_EMA  = construir_pesos_ema()

print(f"[INFO] Trayectoria Λ(t): {N_TOT} muestras   duración total = {T_FINAL:.0f} s   "
      f"T_c (controlador activo) = {T_CONTROL:.0f} s")
print(f"[INFO] Filtro EMA: ventana = {VENTANA_EMA}   α = {ALFA_EMA}   "
      f"(ganancia en régimen permanente = "
      f"{1 - (1 - ALFA_EMA) ** VENTANA_EMA:.4f})")


# =============================================================================
# SIMULACIÓN EN LOTE — PID PROPORCIONAL SOBRE LA TRAYECTORIA
# =============================================================================

@torch.no_grad()
def simular_pid_simple(poblacion, cov, *, ref_bis=None, bis_ini=None,
                        T_sim=N_TOT, T_ven=T_VEN, dt=DT,
                        ppf_max=PPF_MAX, remi_max=REMI_MAX,
                        pesos_ema=None, n_control=N_CTRL):
    """
    Simula el lazo de control del PID proporcional, con filtrado por media
    exponencial móvil, para la trayectoria de referencia variable en el tiempo.
    

    Cada elemento del lote posee sus propias covariables, lo que permite
    simular a toda una cohorte de pacientes —cada uno con su propio conjunto
    de hiperparámetros— en una única llamada.

    Ley de control:
        e(t)         = BIS(t) − Λ(t)
        u_ppf_crudo  = clamp(Kp·e + Ki·∫e·dt + Kd·Δe/Δt, 0, ppf_max)
        u_remi_crudo = clamp(γ · u_ppf_crudo, 0, remi_max)

    Filtrado previo al ingreso a la planta:
        u_ppf  = EMA_ventana(u_ppf_crudo)
        u_remi = EMA_ventana(u_remi_crudo)

    Fases de operación:
        * Sedación y mantenimiento (t < n_control): lazo cerrado.
        * Recuperación (t ≥ n_control): el PID se desactiva y la señal cruda
          de control se anula; la planta evoluciona en lazo abierto. El filtro
          EMA conserva su dinámica, por lo que la infusión introducida a la
          planta decae a cero en VENTANA_EMA muestras.

    Parámetros
    ----------
    poblacion : (N, 4) — [Kp, Ki, Kd, gamma] por paciente
    cov       : (N, 4) o (1, 4) — covariables
    n_control : número de muestras en que el controlador permanece activo

    Retorna
    -------
    ppf_filt_hist, remi_filt_hist, bis_hist — cada uno de forma (N, T_sim)
    """
    if ref_bis   is None: ref_bis   = REF_TENSOR
    if pesos_ema is None: pesos_ema = PESOS_EMA
    if bis_ini   is None: bis_ini   = float(ref_bis[0].item())

    N   = poblacion.shape[0]
    Kp, Ki, Kd, gam = (poblacion[:, 0], poblacion[:, 1],
                        poblacion[:, 2], poblacion[:, 3])
    cov_lote = cov if cov.shape[0] == N else cov.expand(N, -1)

    ppf_raw_h   = torch.zeros(N, T_sim,     dtype=torch.float32, device=dispositivo)
    remi_raw_h  = torch.zeros(N, T_sim,     dtype=torch.float32, device=dispositivo)
    ppf_filt_h  = torch.zeros(N, T_sim,     dtype=torch.float32, device=dispositivo)
    remi_filt_h = torch.zeros(N, T_sim,     dtype=torch.float32, device=dispositivo)
    bis_h       = torch.zeros(N, T_sim + 1, dtype=torch.float32, device=dispositivo)
    bis_h[:, 0] = bis_ini

    integr = torch.zeros(N, device=dispositivo)   # Integrador libre (sin anti-windup)
    e_prev = torch.zeros(N, device=dispositivo)
    ceros_ctrl = torch.zeros(N, device=dispositivo)

    for t in range(T_sim):
        if t < n_control:
            # ── Sedación y mantenimiento: lazo cerrado ─────────────────────────
            e      = bis_h[:, t] - ref_bis[t]
            integr = integr + e * dt
            d_e    = (e - e_prev) / dt
            e_prev = e.clone()

            u_ppf_crudo  = (Kp * e + Ki * integr + Kd * d_e).clamp(0.0, ppf_max)
            u_remi_crudo = (gam * u_ppf_crudo).clamp(0.0, remi_max)
        else:
            # ── Recuperación: PID desactivado, planta en lazo abierto ──────────
            u_ppf_crudo  = ceros_ctrl
            u_remi_crudo = ceros_ctrl

        ppf_raw_h[:, t]  = u_ppf_crudo
        remi_raw_h[:, t] = u_remi_crudo

        # Filtrado EMA antes de introducirse a la planta
        ppf_filt_h[:, t]  = filtro_ema_ventana(ppf_raw_h,  t, pesos_ema)
        remi_filt_h[:, t] = filtro_ema_ventana(remi_raw_h, t, pesos_ema)

        # ── Ventanas de entrada del modelo predictor, construidas a partir de la
        #    señal filtrada con el relleno y la normalización global empleados
        #    en el entrenamiento (Normalizacion_BIS.py) ─────────────────────────
        inf_t, bis_t = construir_ventanas_simulacion(
            ppf_filt_h, remi_filt_h, bis_h, t, T_ven, ESTADISTICAS
        )
        bis_pred, _ = modelo(inf_t, bis_t, cov_lote)
        bis_h[:, t + 1] = bis_pred

    return ppf_filt_h, remi_filt_h, bis_h[:, 1:]


# =============================================================================
# FUNCIÓN DE ERROR ξ 
# =============================================================================

def calcular_xi(bis_np: np.ndarray) -> np.ndarray:
    """
    Evalúa la función de error ξ, definida como la integral del tiempo
    multiplicado por el valor absoluto del error relativo a la trayectoria de
    referencia, normalizada por el cuadrado de la duración del horizonte en
    que el controlador permanece activo:

              2      ⌠ T_c       | e(t) |
        ξ  = ────  · ⎮      t · │────────│ dt
             T_c²    ⌡ 0        |  Λ(t)  |

    donde e(t) = BIS(t) − Λ(t), Λ(t) es la trayectoria de referencia y T_c es
    la duración conjunta de las fases de sedación y mantenimiento. La fase de
    recuperación, en la que la planta evoluciona en lazo abierto, no interviene
    en ξ, en concordancia con la función de costo empleada en la sintonización
    poblacional.

    El factor de normalización 2/T_c² confiere a ξ un carácter adimensional:
    si el error relativo permaneciera constante e igual a c a lo largo de todo
    el horizonte, se obtendría exactamente ξ = c. En consecuencia, ξ admite
    una interpretación directa como error relativo medio ponderado en el
    tiempo, penalizando con mayor severidad las desviaciones tardías.

    La integral se aproxima numéricamente mediante suma de Riemann con paso DT.

    Parámetros
    ----------
    bis_np : (N, T) — respuesta BIS simulada de cada paciente

    Retorna
    -------
    (N,) — valor de ξ para cada paciente
    """
    ref_ctrl  = LAMBDA_REF[None, :N_CTRL]
    error_rel = np.abs((bis_np[:, :N_CTRL] - ref_ctrl) / ref_ctrl)
    integral  = np.sum(EJE_T[None, :N_CTRL] * error_rel, axis=1) * DT
    return (2.0 / T_CONTROL ** 2) * integral


def evaluar_cohorte(params_np: np.ndarray, df_pac: pd.DataFrame) -> tuple:
    """
    Simula la trayectoria completa para una cohorte de pacientes, cada uno con
    su propio conjunto de hiperparámetros, y evalúa la función de error ξ.

    Parámetros
    ----------
    params_np : (N, 4) — hiperparámetros [Kp, Ki, Kd, gamma] por paciente
    df_pac    : DataFrame de N pacientes

    Retorna
    -------
    xi     : (N,)       — valor de la función de error de cada paciente
    bis_np : (N, N_TOT) — respuesta BIS simulada
    """
    params_t = torch.tensor(params_np, dtype=torch.float32, device=dispositivo)
    cov_t    = hacer_cov_lote(df_pac)
    _, _, bis_t = simular_pid_simple(params_t, cov_t)
    bis_np = bis_t.cpu().numpy()
    return calcular_xi(bis_np), bis_np


[INFO] Trayectoria Λ(t): 570 muestras   duración total = 5700 s   T_c (controlador activo) = 3900 s
[INFO] Filtro EMA: ventana = 12   α = 0.05   (ganancia en régimen permanente = 0.4596)


## Etapa 3: Conformación de los conjuntos (tomada de `KNN_Estimacion_PID.ipynb`)

In [3]:
# =============================================================================
# CARGA DE DATOS Y CONFORMACIÓN DE LOS CONJUNTOS
# =============================================================================
#
#   1. Datos de VALIDACIÓN : registros cuyo RMSE es estrictamente mayor al
#      umbral de descarte. Se descartan del proceso de estimación y se
#      reservan para validación.
#
#      El umbral se determina mediante el criterio de Tukey para valores
#      atípicos superiores:
#
#            UMBRAL = Q3 + 1.5 · RIC ,      RIC = Q3 − Q1
#
#      donde Q1 y Q3 son el primer y el tercer cuartil del RMSE de la base de
#      sintonizaciones. El RMSE de SINTONIZACIONES.csv se evalúa en las fases
#      de sedación y mantenimiento, horizonte en el que actúa el controlador.
#
#   2. Datos de ESTIMACIÓN : registros cuyo RMSE es menor o igual al umbral.
#      Este conjunto se subdivide en:
#
#         · ESPACIO (Universo) — conjunto de referencia del algoritmo de K
#           vecinos más cercanos. Se conforma con los pacientes de MENOR error,
#           ordenados de forma ascendente por RMSE, e imponiendo una
#           composición equilibrada del 50 % de hombres y 50 % de mujeres. Es
#           decir, se seleccionan los hombres con menor error y las mujeres con
#           menor error, en igual número.
#
#         · DATOS DE ESTIMACIÓN — los pacientes restantes, cuyos
#           hiperparámetros serán estimados y evaluados.
#
#   Los espacios de los algoritmos separados por sexo se obtienen filtrando el
#   espacio global, por lo que quedan conformados, respectivamente, por los
#   hombres con menor error y por las mujeres con menor error. De esta manera
#   los tres algoritmos operan sobre particiones consistentes entre sí.

UMBRAL_RMSE_FIJO = None   # ← Umbral fijo opcional; None = criterio de Tukey
FACTOR_TUKEY     = 1.5    # ← Factor del rango intercuartílico
FRAC_ESPACIO     = 0.40   # ← Proporción de los datos destinada al Espacio

# ── Lectura del archivo ───────────────────────────────────────────────────────
print(f"[INFO] Cargando: {RUTA_SINTONIZACIONES}")
try:
    df_crudo = pd.read_csv(RUTA_SINTONIZACIONES)
except FileNotFoundError:
    raise FileNotFoundError(
        f"No se encontró el archivo '{RUTA_SINTONIZACIONES}'. Verifique que "
        f"se encuentre en la misma carpeta del proyecto."
    )

columnas_requeridas = {"Edad", "Peso", "Altura", "Sexo",
                       "Kp", "Ki", "Kd", "gamma", "RMSE"}
faltantes = columnas_requeridas - set(df_crudo.columns)
if faltantes:
    raise ValueError(f"El archivo no contiene las columnas esperadas: {faltantes}. "
                     f"Columnas encontradas: {list(df_crudo.columns)}")

# Codificación numérica del sexo: 'M' → 0, 'F' → 1
df_crudo["Sexo_num"] = (
    df_crudo["Sexo"].astype(str).str.strip().str.upper().map({"M": 0, "F": 1})
)
if df_crudo["Sexo_num"].isna().any():
    raise ValueError("Se encontraron valores de Sexo no reconocidos "
                     "(solo se aceptan 'M' y 'F').")

print(f"[INFO] Registros totales: {len(df_crudo)}")

# ── Umbral de descarte ────────────────────────────────────────────────────────
if UMBRAL_RMSE_FIJO is None:
    q1_rmse, q3_rmse = df_crudo["RMSE"].quantile([0.25, 0.75])
    UMBRAL_RMSE = float(q3_rmse + FACTOR_TUKEY * (q3_rmse - q1_rmse))
    print(f"[INFO] Umbral de descarte (criterio de Tukey): "
          f"Q1 = {q1_rmse:.4f}   Q3 = {q3_rmse:.4f}   "
          f"umbral = Q3 + {FACTOR_TUKEY}·RIC = {UMBRAL_RMSE:.4f}")
else:
    UMBRAL_RMSE = float(UMBRAL_RMSE_FIJO)
    print(f"[INFO] Umbral de descarte fijo: {UMBRAL_RMSE:.4f}")

# ── Partición 1: validación (RMSE > umbral) frente a estimación (RMSE ≤ umbral)
df_validacion = (
    df_crudo[df_crudo["RMSE"] > UMBRAL_RMSE].copy().reset_index(drop=True)
)
df_estimacion_total = (
    df_crudo[df_crudo["RMSE"] <= UMBRAL_RMSE].copy().reset_index(drop=True)
)

print(f"[INFO] Datos de VALIDACIÓN (RMSE > {UMBRAL_RMSE:.4f}) : "
      f"{len(df_validacion)} pacientes")
print(f"[INFO] Datos de ESTIMACIÓN (RMSE ≤ {UMBRAL_RMSE:.4f}) : "
      f"{len(df_estimacion_total)} pacientes")

# ── Partición 2: Espacio equilibrado por sexo, seleccionado por menor RMSE ───
df_h_orden = (df_estimacion_total[df_estimacion_total["Sexo_num"] == 0]
              .sort_values("RMSE", ascending=True))
df_m_orden = (df_estimacion_total[df_estimacion_total["Sexo_num"] == 1]
              .sort_values("RMSE", ascending=True))

n_espacio_objetivo = int(round(FRAC_ESPACIO * len(df_estimacion_total)))
# Se toma el mismo número de hombres y de mujeres, garantizando la
# composición equilibrada del 50 % solicitada.
n_por_sexo = max(1, min(n_espacio_objetivo // 2,
                         len(df_h_orden) - 1, len(df_m_orden) - 1))

idx_espacio = list(df_h_orden.index[:n_por_sexo]) + \
              list(df_m_orden.index[:n_por_sexo])

df_espacio = (df_estimacion_total.loc[idx_espacio]
              .copy().reset_index(drop=True))
df_estimados = (df_estimacion_total.drop(index=idx_espacio)
                .copy().reset_index(drop=True))

print(f"\n[INFO] ESPACIO (referencia)  : {len(df_espacio)} pacientes  "
      f"→ {n_por_sexo} hombres + {n_por_sexo} mujeres (50 % / 50 %)")
print(f"[INFO] DATOS DE ESTIMACIÓN   : {len(df_estimados)} pacientes  "
      f"→ hombres: {(df_estimados['Sexo_num']==0).sum()}   "
      f"mujeres: {(df_estimados['Sexo_num']==1).sum()}")
print(f"       RMSE del Espacio      : mín {df_espacio['RMSE'].min():.4f}   "
      f"máx {df_espacio['RMSE'].max():.4f}")

if len(df_estimados) == 0:
    raise ValueError("No quedaron pacientes en el conjunto de estimación. "
                     "Reduzca el valor de FRAC_ESPACIO.")

# ── Subconjuntos por sexo, derivados de la partición global ──────────────────
df_espacio_h   = df_espacio[df_espacio["Sexo_num"] == 0].reset_index(drop=True)
df_espacio_m   = df_espacio[df_espacio["Sexo_num"] == 1].reset_index(drop=True)
df_estimados_h = df_estimados[df_estimados["Sexo_num"] == 0].reset_index(drop=True)
df_estimados_m = df_estimados[df_estimados["Sexo_num"] == 1].reset_index(drop=True)

print(f"\n[INFO] Subconjunto masculino : espacio {len(df_espacio_h)}  |  "
      f"a estimar {len(df_estimados_h)}")
print(f"[INFO] Subconjunto femenino  : espacio {len(df_espacio_m)}  |  "
      f"a estimar {len(df_estimados_m)}")

# ── Vista previa ──────────────────────────────────────────────────────────────
print("\n" + "=" * 72)
print("  ESPACIO — pacientes de menor RMSE, equilibrado por sexo")
print("=" * 72)
try:
    display(df_espacio.sort_values("RMSE"))
except NameError:
    print(df_espacio.sort_values("RMSE").to_string(index=False))

print("\n" + "=" * 72)
print("  DATOS DE ESTIMACIÓN")
print("=" * 72)
try:
    display(df_estimados.head(12))
except NameError:
    print(df_estimados.head(12).to_string(index=False))


[INFO] Cargando: SINTONIZACIONES.csv
[INFO] Registros totales: 340
[INFO] Umbral de descarte (criterio de Tukey): Q1 = 5.4289   Q3 = 5.7459   umbral = Q3 + 1.5·RIC = 6.2213
[INFO] Datos de VALIDACIÓN (RMSE > 6.2213) : 6 pacientes
[INFO] Datos de ESTIMACIÓN (RMSE ≤ 6.2213) : 334 pacientes

[INFO] ESPACIO (referencia)  : 134 pacientes  → 67 hombres + 67 mujeres (50 % / 50 %)
[INFO] DATOS DE ESTIMACIÓN   : 200 pacientes  → hombres: 100   mujeres: 100
       RMSE del Espacio      : mín 5.0274   máx 5.5865

[INFO] Subconjunto masculino : espacio 67  |  a estimar 100
[INFO] Subconjunto femenino  : espacio 67  |  a estimar 100

  ESPACIO — pacientes de menor RMSE, equilibrado por sexo


,Edad,Peso,Altura,Sexo,Kp,Ki,Kd,gamma,RMSE,Sexo_num
67,75.0,35.50,143.4,F,0.003094,9.100000e-07,0.500000,1.149734,5.027434,1
68,70.0,52.30,140.2,F,0.002894,1.000000e-06,0.500000,0.728709,5.060834,1
69,76.0,42.10,151.2,F,0.002731,8.300000e-07,0.479907,2.141863,5.093515,1
70,70.0,46.90,147.3,F,0.003002,8.700000e-07,0.478317,1.129349,5.119910,1
71,86.0,63.45,153.0,F,0.002744,7.200000e-07,0.500000,2.808317,5.133771,1
...,...,...,...,...,...,...,...,...,...,...
62,61.0,70.50,161.3,M,0.005134,8.300000e-07,0.500000,1.154774,5.571923,0
63,62.0,60.70,166.2,M,0.005017,6.700000e-07,0.500000,1.766852,5.577211,0
64,63.0,64.70,169.7,M,0.003817,8.700000e-07,0.398346,1.409008,5.582044,0
65,74.0,62.45,162.6,M,0.004146,9.000000e-07,0.479697,0.498486,5.583477,0



  DATOS DE ESTIMACIÓN


,Edad,Peso,Altura,Sexo,Kp,Ki,Kd,gamma,RMSE,Sexo_num
0,46.0,81.80,174.5,M,0.003692,1.350000e-06,0.500000,1.095321,5.741612,0
1,38.0,62.00,175.0,M,0.005009,1.010000e-06,0.500000,0.777356,5.801717,0
2,58.0,82.80,166.4,M,0.019883,5.800000e-07,0.500000,0.628231,5.947594,0
3,73.0,72.30,172.6,M,0.005035,7.900000e-07,0.500000,1.636975,5.641539,0
4,57.0,58.00,167.9,M,0.005225,8.900000e-07,0.500000,1.842657,5.643056,0
5,58.0,70.45,182.0,M,0.003295,7.200000e-07,0.482809,2.215868,5.629665,0
6,41.0,84.80,171.4,M,0.004797,8.000000e-07,0.500000,1.266457,5.813962,0
7,55.0,51.50,168.8,M,0.004343,8.700000e-07,0.123968,1.015161,5.918763,0
8,76.0,61.50,166.4,M,0.004500,7.900000e-07,0.500000,0.362687,5.863411,0
9,62.0,62.05,160.2,M,0.006352,9.900000e-07,0.500000,1.956855,5.634702,0


## Etapa 4: Funciones del algoritmo de K vecinos más cercanos (tomada de `KNN_Estimacion_PID.ipynb`)

In [4]:
# =============================================================================
# ALGORITMO DE K VECINOS MÁS CERCANOS — FUNCIONES BASE
# =============================================================================
#
# Procedimiento de estimación de los hiperparámetros del PID proporcional:
#
#   1. Los vectores de características del Espacio se normalizan mediante
#      min-max al intervalo [0, 1]. La variable Sexo, por ser categórica
#      binaria, se codifica directamente como 0 y 1 sin reescalamiento.
#
#   2. Cada paciente a estimar se normaliza con esos mismos rangos y se
#      calculan sus distancias de Minkowski respecto a todo el Espacio.
#
#   3. Se seleccionan los K vecinos más cercanos, se toman los inversos de sus
#      distancias (1/dᵢ) y se transforman mediante la función softmax, de modo
#      que los pesos resultantes sumen la unidad.
#
#   4. La estimación es la combinación lineal de los vectores de
#      hiperparámetros de dichos vecinos, ponderada por esos pesos, y se acota
#      finalmente al intervalo admisible.
#
#   5. El desempeño de cada valor del exponente p se evalúa simulando el lazo
#      de control y calculando la función de error ξ.

# Cotas de la estimación final, parámetro a parámetro [Kp, Ki, Kd, γ];
# coinciden con el intervalo de búsqueda del Algoritmo Genético empleado en la
# sintonización poblacional (GANANCIAS_INF/SUP y RHO_INF/SUP)
COTA_INF = np.array([1e-9, 1e-9, 1e-9, 1e-8])   # ← Cotas inferiores
COTA_SUP = np.array([0.5,  0.5,  0.5,  3.0])    # ← Cotas superiores
COLS_HIPER = ["Kp", "Ki", "Kd", "gamma"]   # Hiperparámetros a estimar


def generar_valores_p(p_inicio: float, p_fin: float,
                       p_incremento: float) -> list:
    """
    Genera la lista de exponentes p de la distancia de Minkowski a evaluar,
    a partir del valor inicial, el valor final y el incremento especificados.

    Parámetros
    ----------
    p_inicio     : primer valor del exponente
    p_fin        : último valor del exponente (inclusive)
    p_incremento : paso entre valores consecutivos

    Retorna
    -------
    list de floats con los valores del exponente
    """
    n_pasos = int(round((p_fin - p_inicio) / p_incremento)) + 1
    return [round(p_inicio + i * p_incremento, 10) for i in range(n_pasos)]


def ajustar_normalizador(df_ref: pd.DataFrame, columnas: list) -> dict:
    """
    Calcula los rangos min-max de las variables a partir del Espacio.
    """
    return {c: (float(df_ref[c].min()), float(df_ref[c].max()))
            for c in columnas}


def aplicar_normalizador(df_dat: pd.DataFrame, columnas: list,
                          rangos: dict) -> np.ndarray:
    """
    Construye la matriz de vectores de características normalizados.

    Las variables continuas se reescalan al intervalo [0, 1] con los rangos
    del Espacio. La variable Sexo se emplea directamente con sus valores 0 y 1.
    Si el rango de alguna variable resulta nulo, dicha variable se fija en cero
    para evitar una división entre cero.
    """
    X = np.zeros((len(df_dat), len(columnas)), dtype=np.float64)
    for j, c in enumerate(columnas):
        valores = df_dat[c].to_numpy(dtype=np.float64)
        if c == "Sexo_num":
            X[:, j] = valores          # Categórica binaria: sin reescalamiento
        else:
            v_min, v_max = rangos[c]
            X[:, j] = 0.0 if v_max == v_min else (valores - v_min) / (v_max - v_min)
    return X


def distancia_minkowski(X_ref: np.ndarray, x_consulta: np.ndarray,
                         p: float) -> np.ndarray:
    """
    Distancia de Minkowski de orden p:  d(x,y) = ( Σ |xᵢ − yᵢ|^p )^(1/p)
    """
    return np.sum(np.abs(X_ref - x_consulta[None, :]) ** p, axis=1) ** (1.0 / p)


def pesos_softmax_inversos(distancias: np.ndarray) -> np.ndarray:
    """
    Calcula los pesos de la combinación lineal a partir de los inversos de las
    distancias de los K vecinos, transformados mediante la función softmax:

        wᵢ = exp(1/dᵢ) / Σⱼ exp(1/dⱼ)

    Se sustrae el máximo antes de exponenciar, procedimiento numéricamente
    estable que evita desbordamientos cuando alguna distancia es muy próxima
    a cero.
    """
    inversos   = 1.0 / np.maximum(distancias, 1e-12)
    exponentes = np.exp(inversos - np.max(inversos))
    return exponentes / np.sum(exponentes)


def estimar_hiperparametros_knn(X_espacio: np.ndarray, Y_espacio: np.ndarray,
                                 X_consulta: np.ndarray, k: int, p: float,
                                 cota_inf: np.ndarray = COTA_INF,
                                 cota_sup: np.ndarray = COTA_SUP) -> np.ndarray:
    """
    Estima los hiperparámetros del controlador mediante K vecinos más cercanos
    con ponderación softmax de los inversos de las distancias de Minkowski.

    Parámetros
    ----------
    X_espacio  : (M, D) — vectores normalizados del Espacio
    Y_espacio  : (M, H) — hiperparámetros asociados a cada elemento del Espacio
    X_consulta : (N, D) — vectores normalizados de los pacientes a estimar
    k          : número de vecinos
    p          : orden de la distancia de Minkowski

    Retorna
    -------
    (N, H) — hiperparámetros estimados, acotados al intervalo admisible
    """
    N, H = X_consulta.shape[0], Y_espacio.shape[1]
    Y_est = np.zeros((N, H), dtype=np.float64)
    k_efectivo = min(k, X_espacio.shape[0])

    for i in range(N):
        d = distancia_minkowski(X_espacio, X_consulta[i], p)
        idx_vecinos = np.argsort(d)[:k_efectivo]
        w = pesos_softmax_inversos(d[idx_vecinos])
        Y_est[i] = w @ Y_espacio[idx_vecinos]

    return np.clip(Y_est, cota_inf, cota_sup)


def barrido_exponente_p(X_espacio, Y_espacio, X_consulta, Y_real,
                         df_consulta, k, valores_p, mostrar_progreso=True):
    """
    Ejecuta la estimación para cada valor del exponente p, simula el lazo de
    control resultante y evalúa la función de error ξ.

    Adicionalmente se registra, con carácter informativo, el error absoluto
    medio de cada hiperparámetro respecto a los valores obtenidos previamente
    mediante Algoritmo Genético.

    Retorna
    -------
    tabla     : DataFrame con una fila por valor de p
    registros : dict {p: (Y_est, xi, bis)} con los resultados detallados
    """
    filas, registros = [], {}
    t0 = time.time()

    for n_p, p in enumerate(valores_p, start=1):
        Y_est = estimar_hiperparametros_knn(X_espacio, Y_espacio,
                                             X_consulta, k, p)
        xi, bis = evaluar_cohorte(Y_est, df_consulta)

        fila = {"p": p,
                "xi medio": float(np.mean(xi)),
                "xi desv.est.": float(np.std(xi, ddof=1)) if len(xi) > 1 else np.nan,
                "xi mínimo": float(np.min(xi))}
        # Error absoluto medio por hiperparámetro (informativo)
        for j, c in enumerate(COLS_HIPER):
            fila[f"EAM {c}"] = float(np.mean(np.abs(Y_est[:, j] - Y_real[:, j])))

        filas.append(fila)
        registros[p] = (Y_est, xi, bis)

        if mostrar_progreso:
            transcurrido = time.time() - t0
            eta = transcurrido * (len(valores_p) - n_p) / n_p
            sys.stdout.write(
                f"\r    p = {p:<6.3f}  ({n_p:>3d}/{len(valores_p)})  "
                f"ξ medio = {fila['xi medio']:.6f}   "
                f"[{transcurrido:5.0f}s, ETA {eta:5.0f}s]        ")
            sys.stdout.flush()

    if mostrar_progreso:
        sys.stdout.write("\n")

    return pd.DataFrame(filas), registros


def ejecutar_algoritmo_knn(df_espacio_alg, df_estimados_alg, columnas_vector,
                            k, valores_p, nombre_algoritmo) -> dict:
    """
    Ejecuta el procedimiento completo de un algoritmo: normalización, barrido
    sobre el exponente p, selección del valor con menor ξ medio y presentación
    de los resultados.

    Parámetros
    ----------
    df_espacio_alg   : DataFrame del Espacio del algoritmo
    df_estimados_alg : DataFrame de los pacientes a estimar
    columnas_vector  : variables que conforman el vector de características
    k                : número de vecinos del algoritmo
    valores_p        : lista de exponentes de Minkowski a evaluar
    nombre_algoritmo : denominación del algoritmo

    Retorna
    -------
    dict con: nombre, tabla_p, mejor_p, k, Y_est, xi, bis, df
    """
    print("\n" + "=" * 78)
    print(f"  {nombre_algoritmo}")
    print(f"  Vector de características : {columnas_vector}")
    print(f"  Espacio: {len(df_espacio_alg)} pacientes   |   "
          f"A estimar: {len(df_estimados_alg)} pacientes   |   K = {k}")
    print(f"  Exponentes p evaluados    : {len(valores_p)}  "
          f"(de {valores_p[0]} a {valores_p[-1]})")
    print("=" * 78)

    if len(df_espacio_alg) < k:
        raise ValueError(f"El Espacio ({len(df_espacio_alg)}) es menor que K = {k}.")
    if len(df_estimados_alg) == 0:
        raise ValueError("No hay pacientes que estimar en este algoritmo.")

    rangos = ajustar_normalizador(df_espacio_alg, columnas_vector)
    X_esp  = aplicar_normalizador(df_espacio_alg,  columnas_vector, rangos)
    X_est  = aplicar_normalizador(df_estimados_alg, columnas_vector, rangos)
    Y_esp  = df_espacio_alg[COLS_HIPER].to_numpy(dtype=np.float64)
    Y_real = df_estimados_alg[COLS_HIPER].to_numpy(dtype=np.float64)

    print("\n  Barrido sobre el exponente de la distancia de Minkowski:")
    tabla_p, registros = barrido_exponente_p(
        X_esp, Y_esp, X_est, Y_real, df_estimados_alg, k, valores_p
    )

    print("\n  RESULTADOS POR VALOR DE p")
    print("  " + "─" * 74)
    try:
        display(tabla_p)
    except NameError:
        print(tabla_p.to_string(index=False))

    idx_mejor = int(tabla_p["xi medio"].idxmin())
    mejor_p   = float(tabla_p.loc[idx_mejor, "p"])
    Y_est, xi, bis = registros[mejor_p]

    print(f"\n  ► Mejor exponente: p = {mejor_p}   "
          f"(ξ medio = {tabla_p.loc[idx_mejor, 'xi medio']:.6f})")
    print("  " + "─" * 74)
    print(f"  {'ξ medio':<24}: {np.mean(xi):.6f}")
    print(f"  {'ξ desviación estándar':<24}: "
          f"{np.std(xi, ddof=1) if len(xi) > 1 else float('nan'):.6f}")
    print(f"  {'ξ mínimo / máximo':<24}: {np.min(xi):.6f} / {np.max(xi):.6f}")
    print("=" * 78)

    return {"nombre": nombre_algoritmo, "tabla_p": tabla_p, "mejor_p": mejor_p,
            "k": k, "Y_est": Y_est, "xi": xi, "bis": bis,
            "df": df_estimados_alg.reset_index(drop=True)}


## Etapa 5: Estimación de parámetros y tiempo de cómputo

In [5]:
# =============================================================================
# ESTIMACIÓN DE LOS PARÁMETROS CON LAS TRES ESTRATEGIAS
# =============================================================================
# Variante global de la heurística: K = 2 vecinos y el orden p de la distancia
# de Minkowski con el menor ξ medio en el barrido de KNN_Estimacion_PID.ipynb
K_KNN = 2
barrido_p = pd.read_csv("Barrido_p_KNN.csv")
barrido_global = barrido_p[barrido_p["Variante"] == "Global"]
P_KNN = float(barrido_global.loc[barrido_global["xi_media"].idxmin(), "p"])
print(f"[INFO] Variante global: K = {K_KNN}   p = {P_KNN}")
COLUMNAS_KNN = ["Edad", "Peso", "Altura", "Sexo_num"]

rangos_knn = ajustar_normalizador(df_espacio, COLUMNAS_KNN)
X_espacio  = aplicar_normalizador(df_espacio, COLUMNAS_KNN, rangos_knn)
Y_espacio  = df_espacio[COLS_HIPER].to_numpy(dtype=np.float64)

# Tiempo de cómputo de la estimación para el conjunto de estimación
X_estimados = aplicar_normalizador(df_estimados, COLUMNAS_KNN, rangos_knn)
t0 = time.perf_counter()
Y_knn_estimados = estimar_hiperparametros_knn(X_espacio, Y_espacio, X_estimados, K_KNN, P_KNN)
t_knn = time.perf_counter() - t0
print(f"[INFO] Estimación KNN de {len(X_estimados)} pacientes: {t_knn * 1e3:.2f} ms "
      f"({t_knn / len(X_estimados) * 1e6:.1f} µs por paciente)")

X_deficientes   = aplicar_normalizador(df_validacion, COLUMNAS_KNN, rangos_knn)
Y_knn_deficientes = estimar_hiperparametros_knn(X_espacio, Y_espacio, X_deficientes, K_KNN, P_KNN)

# Controlador de parámetros fijos: mediana del espacio de referencia
mediana_espacio = np.median(Y_espacio, axis=0)
print("[INFO] Parámetros medianos [Kp, Ki, Kd, gamma]:", np.round(mediana_espacio, 5))

estrategias = {
    ("Estimación", "AG"):      (df_estimados[COLS_HIPER].to_numpy(dtype=np.float64), df_estimados),
    ("Estimación", "KNN"):     (Y_knn_estimados, df_estimados),
    ("Estimación", "Mediana"): (np.tile(mediana_espacio, (len(df_estimados), 1)), df_estimados),
    ("Deficientes", "AG"):     (df_validacion[COLS_HIPER].to_numpy(dtype=np.float64), df_validacion),
    ("Deficientes", "KNN"):    (Y_knn_deficientes, df_validacion),
    ("Deficientes", "Mediana"):(np.tile(mediana_espacio, (len(df_validacion), 1)), df_validacion),
}

[INFO] Variante global: K = 2   p = 1.0
[INFO] Estimación KNN de 200 pacientes: 4.56 ms (22.8 µs por paciente)
[INFO] Parámetros medianos [Kp, Ki, Kd, gamma]: [0.00334 0.      0.5     1.56545]


## Etapa 6: Simulación en lazo cerrado y métricas de desempeño

In [6]:
# =============================================================================
# SIMULACIÓN DE LAS TRES ESTRATEGIAS Y CÁLCULO DE MÉTRICAS
# =============================================================================
def metricas_fase(bis_np):
    """
    Calcula, para cada paciente, el RMSE respecto de la trayectoria de
    referencia en las fases de sedación y mantenimiento (horizonte en que el
    controlador permanece activo), el porcentaje del tiempo de mantenimiento
    con el BIS entre 40 y 60, el RMSE durante el mantenimiento y el BIS al
    final de la recuperación en lazo abierto.
    """
    rmse  = np.sqrt(np.mean((bis_np[:, :N_CTRL] - LAMBDA_REF[None, :N_CTRL]) ** 2,
                            axis=1))
    mant  = bis_np[:, N_SED:N_SED + N_MAN]
    pct   = np.mean((mant >= 40) & (mant <= 60), axis=1) * 100
    rmse_mant = np.sqrt(np.mean((mant - BIS_MANT) ** 2, axis=1))
    return rmse, pct, rmse_mant, bis_np[:, -1]


filas, trayectorias = [], {}
for (conjunto, estrategia), (parametros, df_pac) in estrategias.items():
    t0 = time.time()
    xi, bis_np = evaluar_cohorte(parametros, df_pac)
    rmse, pct, rmse_mant, bis_final = metricas_fase(bis_np)
    trayectorias[f"{conjunto}_{estrategia}"] = bis_np
    for i in range(len(df_pac)):
        filas.append(dict(Conjunto=conjunto, Estrategia=estrategia,
                          Edad=df_pac["Edad"].iloc[i], Peso=df_pac["Peso"].iloc[i],
                          Altura=df_pac["Altura"].iloc[i], Sexo=df_pac["Sexo"].iloc[i],
                          xi=xi[i], RMSE=rmse[i], Pct_40_60_mantenimiento=pct[i],
                          RMSE_mantenimiento=rmse_mant[i], BIS_final=bis_final[i]))
    print(f"  {conjunto:12s} {estrategia:8s}  xi = {xi.mean():.4f} ± {xi.std(ddof=1):.4f}   "
          f"RMSE = {rmse.mean():.2f}   [{time.time() - t0:.0f} s]")

resultados = pd.DataFrame(filas)
resultados.to_csv("Resultados_Comparacion_KNN_AG_Mediana.csv", index=False)
np.savez_compressed("Trayectorias_BIS_Comparacion.npz", **trayectorias)
print("[INFO] Resultados guardados en Resultados_Comparacion_KNN_AG_Mediana.csv "
      "y Trayectorias_BIS_Comparacion.npz")

  Estimación   AG        xi = 0.0173 ± 0.0018   RMSE = 5.72   [5 s]
  Estimación   KNN       xi = 0.0162 ± 0.0029   RMSE = 5.46   [5 s]
  Estimación   Mediana   xi = 0.0147 ± 0.0021   RMSE = 5.46   [5 s]
  Deficientes  AG        xi = 0.0194 ± 0.0011   RMSE = 6.50   [3 s]
  Deficientes  KNN       xi = 0.0162 ± 0.0051   RMSE = 5.44   [3 s]
  Deficientes  Mediana   xi = 0.0153 ± 0.0045   RMSE = 5.46   [2 s]
[INFO] Resultados guardados en Resultados_Comparacion_KNN_AG_Mediana.csv y Trayectorias_BIS_Comparacion.npz


## Etapa 7: Tabla resumen y pruebas estadísticas

In [7]:
# =============================================================================
# TABLA RESUMEN (MEDIA ± DESVIACIÓN ESTÁNDAR)
# =============================================================================
METRICAS_COMP = ["xi", "RMSE", "Pct_40_60_mantenimiento", "BIS_final"]
resumen = (resultados.groupby(["Conjunto", "Estrategia"])[METRICAS_COMP]
                     .agg(["mean", "std"]).round(3))
resumen

xi          RMSE        Pct_40_60_mantenimiento  \
                         mean    std   mean    std                    mean   
Conjunto    Estrategia                                                       
Deficientes AG          0.019  0.001  6.499  0.122                  92.269   
            KNN         0.016  0.005  5.440  0.207                  94.491   
            Mediana     0.015  0.004  5.460  0.216                  94.306   
Estimación  AG          0.017  0.002  5.720  0.167                  93.262   
            KNN         0.016  0.003  5.465  0.121                  94.400   
            Mediana     0.015  0.002  5.465  0.144                  94.356   

                               BIS_final         
                          std       mean    std  
Conjunto    Estrategia                           
Deficientes AG          0.369  79.538002  0.598  
            KNN         0.409  80.392998  0.745  
            Mediana     0.421  80.475998  0.954  
Estimación  AG          0.546  80.369003  0.782  
            KNN         0.443  80.530998  0.687  
            Mediana     0.303  80.611000  0.722

In [8]:
# =============================================================================
# COMPARACIONES APAREADAS MEDIANTE LA PRUEBA DE RANGOS CON SIGNO DE WILCOXON
# =============================================================================
COMPARACIONES = {"Estimación": [("KNN", "AG"), ("KNN", "Mediana"), ("AG", "Mediana")],
                 "Deficientes": [("KNN", "AG"), ("KNN", "Mediana")]}
filas = []
for conjunto, pares in COMPARACIONES.items():
    sub = resultados[resultados.Conjunto == conjunto]
    for a, b in pares:
        fila = dict(Conjunto=conjunto, Comparacion=f"{a} frente a {b}")
        for metrica in ["xi", "RMSE"]:
            va = sub.loc[sub.Estrategia == a, metrica].to_numpy()
            vb = sub.loc[sub.Estrategia == b, metrica].to_numpy()
            fila[f"Dif_mediana_{metrica}"] = float(np.median(va - vb))
            fila[f"Valor_p_{metrica}"] = stats.wilcoxon(va, vb).pvalue
            fila[f"Primero_mejor_{metrica}"] = f"{int((va < vb).sum())}/{len(va)}"
        filas.append(fila)
pruebas = pd.DataFrame(filas)
pruebas

,Conjunto,Comparacion,Dif_mediana_xi,Valor_p_xi,Primero_mejor_xi,Dif_mediana_RMSE,Valor_p_RMSE,Primero_mejor_RMSE
0,Estimación,KNN frente a AG,-0.001361,6.440107e-12,147/200,-0.244347,3.231964e-34,195/200
1,Estimación,KNN frente a Mediana,0.001210,4.882952e-28,28/200,-0.024173,3.141076e-01,122/200
2,Estimación,AG frente a Mediana,0.002528,2.416440e-30,11/200,0.246774,5.215949e-34,5/200
3,Deficientes,KNN frente a AG,-0.004854,2.187500e-01,5/6,-1.085945,3.125000e-02,6/6
4,Deficientes,KNN frente a Mediana,0.000900,3.125000e-02,0/6,-0.029441,3.125000e-01,4/6


In [9]:
# =============================================================================
# RAZÓN DE DESEMPEÑO ENTRE LA HEURÍSTICA Y LA SINTONIZACIÓN INDIVIDUAL
# =============================================================================
sub = resultados[resultados.Conjunto == "Estimación"]
xi_knn = sub.loc[sub.Estrategia == "KNN", "xi"].to_numpy()
xi_ag  = sub.loc[sub.Estrategia == "AG", "xi"].to_numpy()
xi_med = sub.loc[sub.Estrategia == "Mediana", "xi"].to_numpy()
print(f"Razón mediana xi_KNN / xi_AG por paciente: {np.median(xi_knn / xi_ag):.3f}")
print(f"xi medio — AG: {xi_ag.mean():.5f}   KNN: {xi_knn.mean():.5f}   "
      f"Mediana: {xi_med.mean():.5f}")
# La fracción de la mejora recuperada solo tiene sentido cuando el AG supera al
# controlador de parámetros medianos
if xi_med.mean() > xi_ag.mean():
    recuperado = (xi_med.mean() - xi_knn.mean()) / (xi_med.mean() - xi_ag.mean()) * 100
    print(f"Fracción de la mejora en xi recuperada por la heurística: {recuperado:.0f} %")
else:
    print("El controlador de parámetros medianos presenta un ξ medio menor o igual "
          "al del AG; la fracción de la mejora recuperada no está definida.")

Razón mediana xi_KNN / xi_AG por paciente: 0.923
xi medio — AG: 0.01730   KNN: 0.01622   Mediana: 0.01470
El controlador de parámetros medianos presenta un ξ medio menor o igual al del AG; la fracción de la mejora recuperada no está definida.
